# Boundary P=4, gamma=0.6 — run split 2 of 3

One of three parallel notebook copies covering the gamma=0.6 boundary
sweep at patch size 4. This copy trains **CI seeds {42, 123, 456, 789, 1011}; then CD seeds {1011}**.

Protocol, model architecture, training schedule, and checkpoint/resume
behaviour are identical to `train_boundary_p4.ipynb`; see that notebook
for the full data-generation, model, and training-loop description. This
copy differs only in its gamma value (0.6) and the run-split assignment
in the next cell.

**Config:** this split's configuration (gamma, seeds, modes) is embedded
in the next cell as a `CFG` literal rather than loaded from an external
config file, so each parallel copy is self-contained. A hash check
guards against an accidental hand-edit after generation (embedded CFG
hash: `57409d96691c20e5`).

**CI note:** an earlier, different-environment run already produced
CI-only reference rows at this gamma value. Those rows cannot substitute
for this split's own CI arm (where assigned): a valid CD/CI comparison
requires both modes trained in the same environment, since the earlier
rows ran under a different attention-kernel implementation. This split's
own CI seeds, where present, are what the paired contrast needs.

**Launch:** new notebook from this file, GPU T4, attach the model-code
dataset. Run All. Progress prints per epoch; a session that reaches its
time budget stops cleanly, and the next Run All resumes from the
previous version's output attached as an input.


In [ ]:
# ── Cell 1: Environment ──────────────────────────────────────────────────────
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# ── Cell 2: Committed-module import with tripwire asserts ─────────────────────
# Model code is NOT inlined. It comes from the attached private Kaggle dataset
# b1-block-attn, whose files are verbatim copies from branch
# revision/tmlr10341. Same tripwires: wrong dataset, extra files, or
# a pre-AMP-fix module version fails fast before anything runs.
import hashlib
import sys

EXPECTED_FILES = {"models.py", "models_cd_block.py"}
AMP_FIX_MARKER = "Allocated from the first encoder output"

INPUT_ROOT = Path("/kaggle/input")
_hits = sorted(INPUT_ROOT.rglob("models_cd_block.py"))
assert len(_hits) == 1, (
    f"Expected exactly one attached dataset containing models_cd_block.py; found {_hits}. "
    "Attach b1-block-attn and nothing else that carries a models_cd_block.py."
)
MODULE_DIR = _hits[0].parent
_py_files = {p.name for p in MODULE_DIR.glob("*.py")}
assert _py_files == EXPECTED_FILES, (
    f"Dataset must contain exactly {sorted(EXPECTED_FILES)}; found {sorted(_py_files)}."
)
_src = (MODULE_DIR / "models_cd_block.py").read_text()
assert AMP_FIX_MARKER in _src, (
    "models_cd_block.py lacks the AMP-fix marker comment — a pre-fix version was uploaded. "
    "Re-copy the file from branch revision/tmlr10341."
)

sys.path.insert(0, str(MODULE_DIR))
import models as M  # noqa: E402

assert callable(getattr(M, "build_model", None)), "models.py must expose build_model()"
for _f in sorted(EXPECTED_FILES):
    _h = hashlib.sha256((MODULE_DIR / _f).read_bytes()).hexdigest()[:16]
    print(f"  {_f}: sha256[:16]={_h}")
print(f"Committed modules loaded from {MODULE_DIR}")

In [ ]:
# ── Cell 3: Embedded run-split configuration ─────────────────────────────────
# This run split's configuration is embedded here rather than read from an
# external config file, so each of the parallel notebook copies for this
# tranche is self-contained. The hash below is fixed at generation time and
# re-derived at runtime from the CFG literal; a mismatch means the literal
# was hand-edited after generation, which would otherwise silently change
# what trains without any visible signal.
CFG = {
    "assignment": 'slice2-tranche3-gamma06-CI-plus-CD',
    "patch_size": 4,
    "batch_size": 128,
    "session_budget_hours": 11.0,
    "runs": [
        {
            "gamma": 0.6,
            "mode": "CI",
            "seeds": [
                42,
                123,
                456,
                789,
                1011
            ]
        },
        {
            "gamma": 0.6,
            "mode": "CD",
            "seeds": [
                1011
            ]
        }
    ],
}
_cfg_hash = __import__("hashlib").sha256(
    __import__("json").dumps(CFG, sort_keys=True).encode()).hexdigest()[:16]
assert _cfg_hash == '57409d96691c20e5', (
    f"Embedded CFG hash {_cfg_hash} != expected '57409d96691c20e5' — this notebook's CFG literal "
    f"was edited after generation; re-derive the expected hash before trusting "
    f"the run list below.")

for _k in ("assignment", "patch_size", "batch_size", "runs"):
    assert _k in CFG, f"CFG missing key: {_k}"
assert CFG["patch_size"] == 4, f"This is the P=4 notebook; CFG says P={CFG['patch_size']}"
assert int(CFG["batch_size"]) == 128, (
    f"Both arms standardise on batch 128 in this environment; "
    f"CFG says {CFG['batch_size']}. Changing it breaks the committed spe "
    f"tripwire and the measured cost pricing.")
SESSION_BUDGET_S = float(CFG.get("session_budget_hours", 11.0)) * 3600.0
for _r in CFG["runs"]:
    for _k in ("gamma", "mode", "seeds"):
        assert _k in _r, f"run entry missing key {_k}: {_r}"
    assert _r["mode"] in ("CI", "CD"), f"Unknown mode in config: {_r['mode']}"
    assert _r["gamma"] == 0.6, (
        f"This is the gamma-0.6 tranche notebook; run entry has gamma={_r['gamma']}")

RUN_LIST: list[tuple[float, str, int]] = [
    (float(r["gamma"]), str(r["mode"]), int(s))
    for r in CFG["runs"] for s in r["seeds"]
]
print(f"Run split: {CFG['assignment']}  (embedded CFG sha256[:16]={_cfg_hash})")
print(f"Run list ({len(RUN_LIST)} runs, in execution order):")
for g, m, s in RUN_LIST:
    print(f"  gamma={g} mode={m} seed={s}")


In [ ]:
# See train_boundary_p4.ipynb for the full protocol description.

N_LEADERS   = 10
N_TOTAL     = 21
PHI         = 0.8
NOISE_STD   = 0.1
N_TIMESTEPS = 20_000

LOOKBACK = 512
PRED_LEN = 96

PATCH_SIZE   = int(CFG["patch_size"])          # 4
PATCH_STRIDE = PATCH_SIZE // 2                 # S = P/2 = 2 (main.tex)
BATCH_SIZE   = int(CFG["batch_size"])          # 128 for both arms in this environment

D_MODEL, N_HEADS, N_LAYERS, DROPOUT = 64, 8, 3, 0.2
LR, WEIGHT_DECAY, GRAD_CLIP = 1e-4, 1e-4, 1.0
WARMUP_EPOCHS, MAX_EPOCHS, PATIENCE = 10, 50, 10

SCHEMA = ["dataset", "C", "rho", "gamma", "patch_size", "mode", "seed",
          "test_mse", "test_mae", "best_epoch", "batch_size",
          "steps_per_epoch", "total_steps"]

# Committed-CSV truths, asserted before any training (results_boundary.csv /
# results_boundary_p4_ci.csv): train windows 13,393; spe 104 at b128 with
# drop_last=True.
EXPECTED_TRAIN_WINDOWS = 13_393
EXPECTED_SPE_B128      = 104


def generate(gamma: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    A = np.zeros((N_TOTAL, N_TOTAL))
    np.fill_diagonal(A, PHI)
    for i in range(N_LEADERS):
        A[i + N_LEADERS, i] = gamma
    X = np.zeros((N_TIMESTEPS, N_TOTAL))
    X[0] = rng.standard_normal(N_TOTAL) * NOISE_STD
    noise = rng.standard_normal((N_TIMESTEPS - 1, N_TOTAL)) * NOISE_STD
    for t in range(1, N_TIMESTEPS):
        X[t] = A @ X[t - 1] + noise[t - 1]
    return X


def split_and_normalise(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    T = len(X)
    n_train = int(T * 0.70)
    n_val   = int(T * 0.10)
    train, val, test = X[:n_train], X[n_train:n_train + n_val], X[n_train + n_val:]
    mean = train.mean(axis=0, keepdims=True)
    std  = train.std(axis=0, keepdims=True) + 1e-8
    return (train - mean) / std, (val - mean) / std, (test - mean) / std


class WindowDataset(Dataset):
    def __init__(self, data: np.ndarray, lookback: int, pred_len: int) -> None:
        self.data     = torch.from_numpy(np.ascontiguousarray(data, dtype=np.float32))
        self.lookback = lookback
        self.pred_len = pred_len
        self.n        = len(data) - lookback - pred_len + 1
        if self.n <= 0:
            raise ValueError(f"Not enough timesteps ({len(data)}).")

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.data[i : i + self.lookback]
        y = self.data[i + self.lookback : i + self.lookback + self.pred_len]
        return x, y


# ── Protocol tripwire: fail in seconds, never 5.6 h into the wrong protocol ──
# gamma is fixed at 0.9 here regardless of this notebook's own assigned gamma
# (0.6): window/step counts depend only on T, LOOKBACK, PRED_LEN, and the
# split fractions, none of which vary with the coupling matrix's off-diagonal
# gamma entries — confirmed by construction of generate() above and verified
# locally at gamma in {0.0, 0.6, 0.9} before this tranche was launched.
_probe_train, _probe_val, _probe_test = split_and_normalise(generate(0.9, 42))
assert len(_probe_train) == 14_000 and len(_probe_val) == 2_000 and len(_probe_test) == 4_000
_probe_ds = WindowDataset(_probe_train, LOOKBACK, PRED_LEN)
assert len(_probe_ds) == EXPECTED_TRAIN_WINDOWS, (
    f"Train windows {len(_probe_ds)} != committed {EXPECTED_TRAIN_WINDOWS}")
_probe_dl = DataLoader(_probe_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
assert len(_probe_dl) == EXPECTED_SPE_B128, (
    f"steps/epoch {len(_probe_dl)} != committed {EXPECTED_SPE_B128} "
    f"(results_boundary_p4_ci.csv, b128, drop_last=True)")
N_PATCHES = (LOOKBACK - PATCH_SIZE) // PATCH_STRIDE + 1
print(f"Protocol tripwires PASS: windows={len(_probe_ds)}, spe(b{BATCH_SIZE})={len(_probe_dl)}, "
      f"P={PATCH_SIZE} S={PATCH_STRIDE} n_patches={N_PATCHES} C*N={N_TOTAL * N_PATCHES}")
del _probe_train, _probe_val, _probe_test, _probe_ds, _probe_dl

In [ ]:
# ── Cell 5: Model construction from committed modules ─────────────────────────
def build_arm(mode: str) -> nn.Module:
    return M.build_model(
        mode, seq_len=LOOKBACK, pred_len=PRED_LEN, num_variates=N_TOTAL,
        patch_size=PATCH_SIZE, stride=PATCH_STRIDE, d_model=D_MODEL,
        n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT)


def _n_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())


# Architecture asserts at P=4: head Linear(255*64, 96); CI/CD param equality.
_x = torch.zeros(2, LOOKBACK, N_TOTAL)
_models = {mode: build_arm(mode) for mode in ("CI", "CD")}
for _mode, _m in _models.items():
    _y = _m(_x)
    assert _y.shape == (2, PRED_LEN, N_TOTAL), f"{_mode} wrong output shape: {_y.shape}"
    assert _m.head.in_features == N_PATCHES * D_MODEL, f"{_mode} head wrong: {_m.head}"
    assert _m.head.out_features == PRED_LEN, f"{_mode} head wrong: {_m.head}"
assert _n_params(_models["CI"]) == _n_params(_models["CD"]), "CI/CD param mismatch"
print(f"Arms OK: {_n_params(_models['CI'])} params each; "
      f"head Linear({N_PATCHES * D_MODEL}, {PRED_LEN})")
del _models, _x, _y
free_cuda()


In [ ]:
# ── Cell 6: Training engine with per-epoch atomic checkpoint+resume ──────────
WORK      = Path("/kaggle/working")
OUT_PATH  = WORK / "results_boundary_p4.csv"

# ── Cross-session bootstrap (a fresh session's /kaggle/working is empty) ─────
# If the previous version's output is attached as an input, seed working from
# it: registry first, then any checkpoints. Files already in working win, so
# a same-session rerun never regresses to older state.
_prior_csvs = [p for p in INPUT_ROOT.rglob("results_boundary_p4.csv")]
if not OUT_PATH.exists() and _prior_csvs:
    assert len(_prior_csvs) == 1, (
        f"Multiple prior registries attached: {_prior_csvs}. Detach all but the "
        f"latest version's output.")
    shutil.copy(_prior_csvs[0], OUT_PATH)
    print(f"[bootstrap] registry seeded from {_prior_csvs[0]}")
_prior_ckpts: dict[str, list[Path]] = {}
for _p in INPUT_ROOT.rglob("ckpt_P*_*.pt"):
    _prior_ckpts.setdefault(_p.name, []).append(_p)
for _name, _srcs in sorted(_prior_ckpts.items()):
    assert len(_srcs) == 1, (
        f"Checkpoint {_name} found in multiple attached inputs: {_srcs}. "
        f"Detach all but the latest version's output.")
    if not (WORK / _name).exists():
        shutil.copy(_srcs[0], WORK / _name)
        print(f"[bootstrap] {_name} seeded from {_srcs[0]}")

# ── Session budget (12 h cap kills a commit run WITH NO SAVED OUTPUT) ────────
# Stop launching epochs once the next one cannot fit; the version then saves
# its checkpoints and registry, and the next session resumes from them.
SESSION_T0 = time.time()
# Conservative per-epoch seeds (CD 915.7 s/ep, CI 87.7 s/ep at b128),
# replaced by the session's own measured maximum as epochs complete.
EPOCH_EST_S = {"CD": 1000.0, "CI": 150.0}
_BUDGET_MARGIN_S = 900.0    # test eval + checkpoint + version-save overhead


def _out_of_budget(mode: str) -> bool:
    elapsed = time.time() - SESSION_T0
    return elapsed + EPOCH_EST_S[mode] + _BUDGET_MARGIN_S > SESSION_BUDGET_S


def _atomic_torch_save(obj: dict, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def _atomic_csv_save(rows: list[dict], path: Path) -> None:
    tmp = path.with_suffix(".csv.tmp")
    pd.DataFrame(rows, columns=SCHEMA).to_csv(tmp, index=False)
    os.replace(tmp, path)


def _ckpt_path(gamma: float, mode: str, seed: int) -> Path:
    return WORK / f"ckpt_P{PATCH_SIZE}_g{gamma}_{mode}_s{seed}.pt"


def _rng_capture() -> dict:
    return {
        "py":    random.getstate(),
        "np":    np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda":  torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def _rng_restore(st: dict) -> None:
    random.setstate(st["py"])
    np.random.set_state(st["np"])
    torch.set_rng_state(st["torch"].cpu() if torch.is_tensor(st["torch"]) else st["torch"])
    if st["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() if torch.is_tensor(t) else t for t in st["cuda"]])


@torch.no_grad()
def _evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    # Committed boundary convention: eval under autocast, element-mean metrics.
    model.eval()
    mse = mae = n = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE.type == "cuda")):
            pred = model(xb)
        mse += nn.functional.mse_loss(pred.float(), yb, reduction="sum").item()
        mae += nn.functional.l1_loss(pred.float(),  yb, reduction="sum").item()
        n   += yb.numel()
    return mse / n, mae / n


def _cosine_warmup(optimizer, epoch: int) -> None:
    # Committed boundary schedule: linear warmup, cosine decay to 1e-6 floor.
    min_lr = 1e-6
    if epoch < WARMUP_EPOCHS:
        lr = LR * (epoch + 1) / WARMUP_EPOCHS
    else:
        progress = (epoch - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS)
        lr = min_lr + 0.5 * (LR - min_lr) * (1 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg["lr"] = lr


def train_one(gamma: float, mode: str, seed: int) -> dict:
    """One full run at the boundary protocol, resumable at epoch granularity.

    Checkpoint invariant: the file on disk always describes a state at an
    epoch BOUNDARY (written after val + early-stop bookkeeping). A resumed
    session restores parameters, optimiser, scaler, RNG streams, and
    early-stop state, then continues with the next epoch. The shuffle
    permutation of epoch e is drawn from a dedicated generator seeded as
    f(seed, e), so an epoch is identical whether or not a resume preceded it.
    A recorded stop (early stopping or a non-finite epoch) is honoured on
    resume: the loop is skipped and the run finalises from its best state.

    Returns the result row, or None if the session budget was reached before
    the run could finish (checkpoint stays on disk for the next session).
    """
    ckpt_file = _ckpt_path(gamma, mode, seed)
    fingerprint = {"gamma": gamma, "mode": mode, "seed": seed,
                   "P": PATCH_SIZE, "batch": BATCH_SIZE}

    set_seed(seed)
    train_d, val_d, test_d = split_and_normalise(generate(gamma, seed))
    train_ds = WindowDataset(train_d, LOOKBACK, PRED_LEN)
    val_dl   = DataLoader(WindowDataset(val_d, LOOKBACK, PRED_LEN),
                          batch_size=BATCH_SIZE * 4, shuffle=False)
    test_dl  = DataLoader(WindowDataset(test_d, LOOKBACK, PRED_LEN),
                          batch_size=BATCH_SIZE * 4, shuffle=False)

    model  = build_arm(mode).to(DEVICE)
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    criterion = nn.MSELoss()
    spe = None

    start_epoch, best_val, best_epoch = 0, float("inf"), 0
    best_total_steps, patience_ctr, steps_so_far = 0, 0, 0
    best_sd = None

    if ckpt_file.exists():
        ck = torch.load(ckpt_file, map_location="cpu", weights_only=False)
        assert ck["fingerprint"] == fingerprint, (
            f"Checkpoint {ckpt_file.name} fingerprint {ck['fingerprint']} does not match "
            f"this run {fingerprint}; refusing to load.")
        model.load_state_dict(ck["model_sd"])
        opt.load_state_dict(ck["opt_sd"])
        scaler.load_state_dict(ck["scaler_sd"])
        _rng_restore(ck["rng"])
        start_epoch      = ck["epoch_done"]          # epochs fully completed
        best_val         = ck["best_val"]
        best_epoch       = ck["best_epoch"]
        best_total_steps = ck["best_total_steps"]
        patience_ctr     = ck["patience_ctr"]
        steps_so_far     = ck["steps_so_far"]
        best_sd          = ck["best_sd"]
        if ck.get("stopped", False):
            start_epoch = MAX_EPOCHS    # honour the recorded stop: finalise only
        print(f"    [resume] {ckpt_file.name}: {start_epoch if start_epoch <= MAX_EPOCHS else MAX_EPOCHS}"
              f" epochs done, best_val={best_val:.6f} @ epoch {best_epoch}"
              f"{'  [stopped]' if ck.get('stopped', False) else ''}", flush=True)

    stopped  = False
    train_dl = None    # stays None when the loop is skipped (recorded stop)
    for epoch in range(start_epoch, MAX_EPOCHS):
        if _out_of_budget(mode):
            print(f"    [budget] {SESSION_BUDGET_S/3600:.1f} h session budget cannot fit "
                  f"another {mode} epoch; stopping cleanly (checkpoint on disk).", flush=True)
            return None
        t0 = time.time()
        _cosine_warmup(opt, epoch)
        shuffle_gen = torch.Generator()
        shuffle_gen.manual_seed(1_000_003 * seed + epoch)
        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=True, generator=shuffle_gen)
        if spe is None:
            spe = len(train_dl)

        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=(DEVICE.type == "cuda")):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            steps_so_far += 1

        val_mse, _ = _evaluate(model, val_dl)
        if not math.isfinite(val_mse):
            print(f"    [warn] non-finite val at epoch {epoch + 1}; stopping run.", flush=True)
            stopped = True
        elif val_mse < best_val:
            best_val, best_epoch = val_mse, epoch + 1
            best_total_steps     = steps_so_far
            patience_ctr         = 0
            best_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                stopped = True

        _atomic_torch_save({
            "fingerprint": fingerprint, "epoch_done": epoch + 1,
            "model_sd": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "opt_sd": opt.state_dict(), "scaler_sd": scaler.state_dict(),
            "rng": _rng_capture(), "best_val": best_val, "best_epoch": best_epoch,
            "best_total_steps": best_total_steps, "patience_ctr": patience_ctr,
            "steps_so_far": steps_so_far, "best_sd": best_sd, "stopped": stopped,
        }, ckpt_file)
        epoch_s = time.time() - t0
        EPOCH_EST_S[mode] = max(EPOCH_EST_S[mode], epoch_s)
        print(f"    epoch {epoch + 1:2d}/{MAX_EPOCHS}  val={val_mse:.6f}  "
              f"best={best_val:.6f}@{best_epoch}  ({epoch_s:.0f}s)", flush=True)
        if stopped:
            break

    if best_sd is None:
        ckpt_file.unlink(missing_ok=True)   # never resume a dead state forever
        raise RuntimeError(
            f"No best state for gamma={gamma} mode={mode} seed={seed} "
            f"(all epochs non-finite). Run not recorded; checkpoint removed.")
    model.load_state_dict(best_sd)
    test_mse, test_mae = _evaluate(model, test_dl)

    row = {"dataset": "leader_follower_var1", "C": N_TOTAL, "rho": 0.5,
           "gamma": gamma, "patch_size": PATCH_SIZE, "mode": mode, "seed": seed,
           "test_mse": round(test_mse, 8), "test_mae": round(test_mae, 8),
           "best_epoch": best_epoch, "batch_size": BATCH_SIZE,
           "steps_per_epoch": spe if spe is not None else len(train_ds) // BATCH_SIZE,
           "total_steps": best_total_steps}

    del model, opt, scaler, train_dl, val_dl, test_dl, best_sd
    free_cuda()
    ckpt_file.unlink(missing_ok=True)   # completed: registry row is the record
    return row

In [ ]:
# ── Cell 7: Main sweep — idempotent registry, config-ordered ──────────────────
def _key(gamma: float, mode: str, seed: int) -> tuple:
    return (round(float(gamma), 4), str(mode), int(seed))


if OUT_PATH.exists() and OUT_PATH.stat().st_size > 100:
    _existing = pd.read_csv(OUT_PATH)
    assert list(_existing.columns) == SCHEMA, (
        f"{OUT_PATH.name} columns {list(_existing.columns)} != expected {SCHEMA}; "
        f"refusing to resume (a rewrite would silently drop or reorder columns).")
    results = _existing.to_dict("records")
    done = {_key(r["gamma"], r["mode"], r["seed"]) for r in results}
    print(f"Registry: {len(done)}/{len(RUN_LIST)} assigned runs already complete.")
else:
    results, done = [], set()

fails = []
for idx, (gamma, mode, seed) in enumerate(RUN_LIST, 1):
    key = _key(gamma, mode, seed)
    if key in done:
        print(f"[{idx}/{len(RUN_LIST)}] SKIP gamma={gamma} mode={mode} seed={seed}")
        continue
    print(f"[{idx}/{len(RUN_LIST)}] gamma={gamma} mode={mode} seed={seed}", flush=True)
    t0 = time.time()
    try:
        row = train_one(gamma, mode, seed)
    except Exception as exc:
        free_cuda()
        print(f"  FAILED: {type(exc).__name__}: {exc}", flush=True)
        fails.append((gamma, mode, seed, repr(exc)))
        continue
    if row is None:
        print("  Session budget reached. Next session: new version, attach THIS "
              "version's output as an input, Run All to resume.", flush=True)
        break
    results.append(row)
    done.add(key)
    _atomic_csv_save(results, OUT_PATH)
    print(f"  DONE mse={row['test_mse']:.4f} epoch={row['best_epoch']} "
          f"({(time.time() - t0) / 3600:.2f} h)", flush=True)

print(f"\nSession end: {len(done)}/{len(RUN_LIST)} runs complete -> {OUT_PATH}")
if fails:
    print(f"{len(fails)} failures (rerun Run All to retry):")
    for f in fails:
        print(" ", f)


In [ ]:
# ── Cell 8: Sanity summary ────────────────────────────────────────────────────
if OUT_PATH.exists():
    df = pd.read_csv(OUT_PATH)
    print(df.to_string(index=False))
    if {"CI", "CD"} <= set(df["mode"]):
        piv = (df.groupby(["gamma", "mode"])["test_mse"].mean().unstack())
        if {"CI", "CD"} <= set(piv.columns):
            piv["CD/CI"] = piv["CD"] / piv["CI"]
        print()
        print(piv.to_string())
else:
    print("No results yet.")


In [ ]:
# ── Cell 9: Download copy, tagged by this run split's own label ──────────────
# Tagged by the run-split label rather than a literal "complete" once this
# split's own runs finish: "complete" would collide in meaning with the
# canonical merged-tranche filename, which uses that same word for a
# different thing — the full tranche across every split, not one split's
# contribution. The label is unique per split by construction, so this
# name can never collide with another split's output or the merged file.
from IPython.display import FileLink

_pending = [(g, m, s) for (g, m, s) in RUN_LIST if _key(g, m, s) not in done]
_dl = WORK / f"results_boundary_p4_{CFG['assignment']}.csv"
shutil.copy(OUT_PATH, _dl)
_resume_msg = _pending[0] if _pending else "none, this split is finished"
print(f"{len(done)}/{len(RUN_LIST)} complete; resume point: {_resume_msg}")
print(f"Downloaded as: {_dl.name}")
FileLink(_dl.name)
